## 一、整体流程

```
加载数据 → 探索数据 → 清洗 → 加新列 → 过滤 → 处理 HTML
    → 加速处理 → 处理变长 → 转 Pandas → 划分验证集 → 保存
```

---

## 二、逐环节代码

### 1. 加载 TSV

```python
from datasets import load_dataset

data_files = {
    "train": "drugsComTrain_raw.tsv",
    "test": "drugsComTest_raw.tsv"
}
drug_dataset = load_dataset("csv", data_files=data_files, delimiter="\t")
```

> **TSV = 用 `\t` 分隔的 CSV**，用 `delimiter="\t"` 指定。

---

### 2. 探索数据：`shuffle` + `select`

```python
drug_sample = drug_dataset["train"].shuffle(seed=42).select(range(1000))
drug_sample[:3]
```

> **先随机打乱，再取前 1000 条**——快速了解数据长什么样。

**发现的问题：**
- `Unnamed: 0` 像患者 ID
- `condition` 大小写混乱
- 评论有 `\r\n` 和 HTML 字符（`&#039;`）

---

### 3. 验证猜想：`unique` + `assert`

```python
for split in drug_dataset.keys():
    assert len(drug_dataset[split]) == len(drug_dataset[split].unique("Unnamed: 0"))
```

> **验证"每行 ID 唯一" → 确认是患者 ID。**

---

### 4. 重命名列：`rename_column`

```python
drug_dataset = drug_dataset.rename_column("Unnamed: 0", "patient_id")
```

> **`DatasetDict` 的方法，自动应用到所有 split。**

---

### 5. 过滤 None：`filter`

```python
# 先发现报错：'NoneType' object has no attribute 'lower'
drug_dataset = drug_dataset.filter(lambda x: x["condition"] is not None)
```

> **`filter` 用 lambda，返回 `True` 保留、`False` 丢弃。**

---

### 6. 规范标签：`map`

```python
def lowercase_condition(example):
    return {"condition": example["condition"].lower()}

drug_dataset = drug_dataset.map(lowercase_condition)
```

---

### 7. 加新列：`map` 返回新 key

```python
def compute_review_length(example):
    return {"review_length": len(example["review"].split())}

drug_dataset = drug_dataset.map(compute_review_length)
```

> **返回的 key 不在原数据集里 → 自动创建新列。**
>
> **另一种加列方式：`Dataset.add_column()`**

---

### 8. 排序看极端：`sort`

```python
drug_dataset["train"].sort("review_length")[:3]
```

> **看到有的评论只有 1 个词。**

---

### 9. 过滤短评论：`filter`

```python
drug_dataset = drug_dataset.filter(lambda x: x["review_length"] > 30)
```

> **删掉少于 30 词的评论（约 15%）。**

---

### 10. 解码 HTML：`map` + `html.unescape`

```python
import html

# 逐条
drug_dataset = drug_dataset.map(lambda x: {"review": html.unescape(x["review"])})

# 批量（快）
drug_dataset = drug_dataset.map(
    lambda x: {"review": [html.unescape(o) for o in x["review"]]},
    batched=True
)
```

---

### 11. 加速：`batched=True` + `num_proc`

**性能对比：**

| 选项 | 快速 tokenizer | 慢速 tokenizer |
|------|---------------|---------------|
| `batched=True` | **10.8s** | 4min41s |
| `batched=False` | 59.2s | 5min3s |
| `batched=True, num_proc=8` | 6.52s | **41.3s** |

**结论：**
- 快速 tokenizer（默认）+ `batched=True` → 最优
- **别在快速 tokenizer 上加 `num_proc`**（Rust 已并行）
- 慢速 tokenizer + `num_proc=8` → 提速明显

---

### 12. 处理变长：`return_overflowing_tokens`

```python
def tokenize_and_split(examples):
    return tokenizer(
        examples["review"],
        truncation=True,
        max_length=128,
        return_overflowing_tokens=True,   # ← 长文本切多段
    )
```

> **问题**：行数从 1000 变 1463 → 原列和新列长度不一致 → **Arrow 报错**。

**两种解决方案：**

#### 方案 1：删原列

```python
tokenized_dataset = drug_dataset.map(
    tokenize_and_split, batched=True,
    remove_columns=drug_dataset["train"].column_names
)
```

#### 方案 2：用映射保留原列

```python
def tokenize_and_split(examples):
    result = tokenizer(..., return_overflowing_tokens=True)
    sample_map = result.pop("overflow_to_sample_mapping")   # 新片段→原样本的映射
    for key, values in examples.items():
        result[key] = [values[i] for i in sample_map]       # 按映射复制原列
    return result

tokenized_dataset = drug_dataset.map(tokenize_and_split, batched=True)
```

> **`overflow_to_sample_mapping = [0,0,0,1,2,2]`** → 按这个映射复制原列，让长度一致。

---

### 13. 转 Pandas：`set_format`

```python
drug_dataset.set_format("pandas")
train_df = drug_dataset["train"][:]      # ← 必须 [:] 才拿到 DataFrame
```

**关键理解：**
- `set_format` **只改 `__getitem__` 的返回格式**
- **不改底层数据**（还是 Arrow）
- **不改对象类型**（还是 Dataset）

**用 Pandas 分析：**

```python
frequencies = (train_df["condition"]
    .value_counts()
    .to_frame()
    .reset_index()
    .rename(columns={"index": "condition", "count": "frequency"}))

from datasets import Dataset
freq_dataset = Dataset.from_pandas(frequencies)   # 转回 Dataset
```

**切回 Arrow：**

```python
drug_dataset.reset_format()
```

---

### 14. 划分验证集：`train_test_split`

```python
drug_dataset_clean = drug_dataset["train"].train_test_split(train_size=0.8, seed=42)

# 把默认的 "test" 改名为 "validation"
drug_dataset_clean["validation"] = drug_dataset_clean.pop("test")

# 把原始 test 加进来
drug_dataset_clean["test"] = drug_dataset["test"]
```

> **最终结构：train / validation / test。**

---

### 15. 保存数据集

| 格式 | 方法 |
|------|------|
| Arrow | `dataset.save_to_disk("path")` |
| CSV | `dataset.to_csv("path.csv")` |
| JSON | `dataset.to_json("path.jsonl")` |

**保存 + 加载：**

```python
drug_dataset_clean.save_to_disk("drug-reviews")

from datasets import load_from_disk
drug_dataset_reloaded = load_from_disk("drug-reviews")
```

**JSON 要分 split 存：**

```python
for split, dataset in drug_dataset_clean.items():
    dataset.to_json(f"drug-reviews-{split}.jsonl")

# 加载
data_files = {"train": "...", "validation": "...", "test": "..."}
load_dataset("json", data_files=data_files)
```

---

## 三、本节用到的所有 API 速查

| API | 作用 |
|-----|------|
| `load_dataset("csv", data_files=..., delimiter="\t")` | 加载 TSV |
| `shuffle(seed=42)` | 随机打乱 |
| `select(range(1000))` | 取前 N 条 |
| `unique("列")` | 列去重 |
| `rename_column(a, b)` | 重命名列 |
| `map(fn)` / `map(fn, batched=True)` | 处理 |
| `map(fn, num_proc=8)` | 多进程 |
| `map(fn, remove_columns=[...])` | 删原列 |
| `filter(fn)` | 筛选行 |
| `sort("列")` | 排序 |
| `add_column(name, data)` | 加列 |
| `train_test_split(train_size=0.8)` | 划分 |
| `set_format("pandas")` | 切格式 |
| `reset_format()` | 重置格式 |
| `to_pandas()` / `from_pandas()` | 与 Pandas 互转 |
| `save_to_disk` / `load_from_disk` | Arrow 存读 |
| `to_csv` / `to_json` | 导出 |

---

## 四、一句话总结

> 这一节是 **`datasets` 库的实战操作手册**：
>
> 1. **加载**：TSV/CSV/JSON
> 2. **探索**：`shuffle` + `select` + `unique`
> 3. **清洗**：`filter`（删空值）+ `map`（改格式、解码 HTML）
> 4. **加列**：`map` 返回新 key 或 `add_column`
> 5. **加速**：`batched=True`（必用），`num_proc` 看情况
> 6. **处理变长**：`remove_columns` 或 `overflow_to_sample_mapping`
> 7. **格式转换**：`set_format`（只改出口，不改数据）
> 8. **划分**：`train_test_split`
> 9. **保存**：`save_to_disk` / `to_csv` / `to_json`

---

**记忆点：**

```
加载 → 探索 → 清洗(map/filter) → 加速(batched) → 处理变长
    → 格式转换(set_format) → 划分(train_test_split) → 保存
```

---
